# Neural Networks (MLP) for Regression

__INDEX__
1. [Neural Networks (MLPs)](#neuralnet) <br>
    1.1. [About Neural Networks(MLPs)](#about) <br>
    1.2. [MLPRegressor in Scikit-learn](#scikit)<br>
    1.3. [Why We Use MLPs in This Project](#project)<br>
    1.4. [Limitations](#limitations)<br>

2. [Setup and Data Preparation](#setup)
3. [Experimental Runs](#expruns)<br>
    3.1. [Baseline Model (Preprocessed Data Only)](#baseline) <br>
    3.2. [Baseline Model using feature selection](#baselinefs) <br>
    3.3. [Impact of Removing Previous Owners in Feature-Selected Baseline](#noprevious)<br>
    3.4. [Feature Engineering Without Feature Selection](#fe_nofs)<br>
    3.5. [Model Performance with Log-Target, Age Feature, and Feature Selection](#log_age)<br>
    3.6. [Model with Feature Engineering and Feature Selection](#fe_fs)<br>
4. [Final Configuration](#finalconfig)
5. [Visualization and Final Regards about the Implementation](#visualization)

## 1. Neural Networks (MLPs)
<a id="neuralnet"></a>

### 1.1. About Neural Networks(MLPs)
<a id="about"></a>

Neural networks, more specifically Multi-Layer Perceptrons (MLPs), are flexible models that learn complex relationships by combining multiple non-linear transformations across connected layers. Instead of relying on tree structures or rule-based splits, MLPs model the relationship between input features and the target variable by learning a set of weights optimized jointly.

In the context of our car price prediction task, this is particularly relevant, since the relationship between variables such as brand, model, mileage, age, engine size, and fuel type is highly non-linear and involves multiple interactions. MLPs allow us to explore whether these interactions can be captured more effectively by a high-capacity model.

Training an MLP can be seen as an optimization problem: the model minimizes a loss function by iteratively updating its weights through backpropagation. Unlike boosting algorithms, which sequentially correct residual errors, MLPs adjust all parameters simultaneously to reduce the overall prediction error.

### 1.2. MLPRegressor in Scikit-learn
<a id="scikit"></a>

Scikit-learn provides an implementation of feed-forward neural networks for regression through MLPRegressor, which we use in this project. This implementation allows us to control:

- the network architecture (number of hidden layers and neurons);
- the learning rate and batch size;
- the optimization solver (Adam);
- early stopping to reduce overfitting.

Contrary to tree-based models used earlier in this project, MLPs do not natively support missing values or categorical features. For this reason, we apply a full preprocessing pipeline before training the model, including imputations, categorical resolvers, target encoding, one-hot encoding, and feature scaling. This ensures that the neural network receives well-conditioned numerical inputs.

Given the sensitivity of neural networks to hyperparameters, we rely on Random Search combined with K-Fold cross-validation to evaluate different configurations and assess their robustness across folds.

### 1.3 Why We Use MLPs in This Project
<a id="project"></a>

We include MLPs in our experimental setup to complement tree-based approaches and to evaluate how a fully non-linear, high-capacity model performs on our dataset. By using hidden layers, the model can learn interactions between features that may not be explicitly captured by linear models or shallow trees.

Although neural networks require more careful preprocessing and tuning, they provide a useful comparison point and help us understand whether additional modeling flexibility leads to improved price predictions in our setting.

### 1.4. Limitations
<a id="limitations"></a>
Neural networks take longer to train, are sensitive to scaling, and may overfit if the architecture is too large or if early stopping is not used.

## 2. Setup and Preparation
<a id="setup"></a>

In [27]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, ParameterSampler
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, median_absolute_error, mean_absolute_percentage_error
import time
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
import os
import random
import logging
from IPython.display import display

In [28]:
def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def bias(y_true, y_pred) -> float:
    return float(np.mean(np.asarray(y_true) - np.asarray(y_pred)))

def get_random_configs(n_iter: int):
    configs = []
    keys = list(param_distributions.keys())

    for i in range(n_iter):
        params = {k: random.choice(list(param_distributions[k])) for k in keys}

        name = (
            f"R{i}"
            f"_n{params['n_estimators']}"
            f"_ms{params['max_samples']}"
            f"_mf{params['max_features']}"
            f"_d{params['tree_max_depth']}"
            f"_leaf{params['tree_min_samples_leaf']}"
            f"_split{params['tree_min_samples_split']}"
        )

        configs.append((name, params))

    return configs


In [30]:
# All of our preprocessing helper functions are in this notebook
%run 05_0_preproc_helpers.ipynb  

# this is the target column - the value we want to predict 
TARGET_COL = "price"  

# separate features and target variable from the full training datase
y = full_train_dataset[TARGET_COL].copy()
X = full_train_dataset.drop(columns=[TARGET_COL]).copy()

# at this point, this are all the features in use 
categorical_features = ['Brand', 'model', 'transmission', 'fuelType']              
numeric_features = ['year', 'mileage', 'engineSize', 'tax', 'mpg', 'previousOwners']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Num features:", numeric_features)
print("Cat features:", categorical_features)

N_SPLITS = 8 # the number of folds for K-Fold cross-validation
RANDOM_STATE = 42 # seed to control randomness
# ---------------------------------------------------------
# 0) CONFIGS GERAIS
# ---------------------------------------------------------
N_SPLITS = 8
N_ITER = 10
RANDOM_STATE = 42


X shape: (75973, 10)
y shape: (75973,)
Num features: ['year', 'mileage', 'engineSize', 'tax', 'mpg', 'previousOwners']
Cat features: ['Brand', 'model', 'transmission', 'fuelType']


## 3. Experimental Runs
<a id="expruns"></a>

### 3.1 Baseline Model (Preprocessed Data Only) - 1
<a id="baseline"></a>

adicionar scaling 

In [ ]:
# we will use an 5-fold cross-validation strategy
N_SPLITS = 5 
RANDOM_STATE = 42
# shuffles the data, in order to, at least theoretically, have more balanced folds
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# hyperparameter space that we are going to use to random search
# each configuration will get 1 value from each of these lists
param_distributions = {
    # MLP params
    "hidden_layer_sizes": [(256, 128),(128,64)],
    "learning_rate_init": [0.05, 0.01, 0.005],
    "activation": ["relu"],
    "solver": ["adam"],
    "batch_size": [64, 128],
    "max_iter": [900],
    "learning_rate": ["adaptive"],
    "early_stopping": [True],
    "n_iter_no_change": [25]
}

N_RANDOM_CONFIGS = 10 # this is a short random search, enough for baseline results

# we get N_RANDOM_CONFIGS (10) random parameter combinations from param_distributions 
sampler = ParameterSampler(param_distributions, n_iter=N_RANDOM_CONFIGS, random_state=RANDOM_STATE)

search_results = [] # to store results of each configuration, to store the ranking

# tracking the best configuration according to validation RMSE 
best_rmse = np.inf 
best_config = None 

# to store our fold results
log_path = "NN_2complete_log.txt"
# Note: this section is mainly for organizational purposes and is not particularly relevant 


with open(log_path, "w", encoding="utf-8") as log_file:

    # this function allows printing the messages to the screen and to the log file 
    def log(msg: str):
        print(msg) 
        log_file.write(msg + "\n") 
        log_file.flush() 

    # header for the log file - just for readability and general info
    log("# =============================")
    log(f"# START OF NN SEARCH (DETAILED METRICS: R2, RMSE, MAE, BIAS)")
    log(f"# LOG FILE SAVED TO: {log_path}")
    log("# =============================")
    
    # loop that iterates over each random configuration
    for config_id, params in enumerate(sampler, start=1):
        log("")
        log(f"######## CONFIG {config_id}/{N_RANDOM_CONFIGS} ########") 
        log(f"Parameters: {params}")

        # we are storing the metrics for each fold here
        fold_rmses_val, fold_maes_val, fold_r2s_val, fold_bias_val = [], [], [], []
        fold_rmses_tr, fold_maes_tr, fold_r2s_tr, fold_bias_tr = [], [], [], []
        fold_med_ae = []

        # for each configuration, we do K-Fold cross-validation
        for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
            
            # data split for the current fold
            X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
            y_train, y_val = y.iloc[train_idx].copy(), y.iloc[val_idx].copy()

            log(f"\n[C{config_id}|F{fold}] Processing fold...")


            # -> Preprocessing Steps 
            # fit is only done with the training data 
            # we aplly the transformations (some obtained through fit) to both train and validation sets
            # all preprocessing functions are explained before, here we just apply them in order
            year_state = fit_year_median(X_train, year_col="year", model_col="model")
            X_train = transform_year_with_model_median(X_train, state=year_state)
            X_val   = transform_year_with_model_median(X_val,   state=year_state)

            mileage_state = fit_mileage_imputer(X_train, "mileage", True)
            X_train = transform_mileage_imputer(X_train, mileage_state)
            X_val   = transform_mileage_imputer(X_val,   mileage_state)

            engine_state = fit_engine_size_imputer(X_train, engine_col="engineSize")
            X_train = transform_engine_size_imputer(X_train, state=engine_state)
            X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

            X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
            X_val   = transform_tax_custom_rules(X_val, "tax", "year", "fuelType", "engineSize")

            mpg_state = fit_mpg_imputer(X_train, mpg_col="mpg", do_abs=True)
            X_train = transform_mpg_imputer(X_train, state=mpg_state)
            X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

            owners_state = fit_previous_owners_imputer(X_train, "previousOwners", "year", "mileage")
            X_train = transform_previous_owners_imputer(X_train, state=owners_state)
            X_val   = transform_previous_owners_imputer(X_val,   state=owners_state)

            brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
            X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
            X_val, _, _   = transform_ambiguous_brands(X_val, brand_state)

            model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
            X_train, _, _ = transform_invalid_models(X_train, model_state)
            X_val, _, _   = transform_invalid_models(X_val, model_state)

            transm_state = fit_transmission_resolver(X_train, valid_transmissions)
            X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
            X_val, _, _   = transform_transmission_resolver(X_val, transm_state)
            
            fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
            X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
            X_val, _, _   = transform_fueltype_resolver(X_val, fuel_state)


            # -> Feature Encoding
            # applied to categorical features only
            # we separate high cardinality and low cardinality categorical features
            high_card_features = ["Brand", "model"] # target encoding for these
            low_card_features  = [c for c in categorical_features if c not in high_card_features]

            te = MyTargetEncoder(smoothing=5)
            te.fit(X_train[high_card_features], y_train) # fit on training data
            X_train_high = te.transform(X_train[high_card_features])
            X_val_high   = te.transform(X_val[high_card_features])

            ohe = MyOneHotEncoder()
            ohe.fit(X_train[low_card_features]) # fit on training data
            X_train_low = ohe.transform(X_train[low_card_features])
            X_val_low   = ohe.transform(X_val[low_card_features])

            # rejoining the categorical features (that are now encoded)
            X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
            X_val_cat   = pd.concat([X_val_high,   X_val_low],   axis=1)

            # joining numerical and categorical (encoded) features 
            X_train_final = pd.concat([X_train[numeric_features], X_train_cat], axis=1)
            X_val_final   = pd.concat([X_val[numeric_features],   X_val_cat],   axis=1)

            scaler = StandardScaler()
            X_train_final[numeric_features] = scaler.fit_transform(X_train_final[numeric_features])
            X_val_final[numeric_features]   = scaler.transform(X_val_final[numeric_features])

            # here, we log the features name, to keep track of what is being used from the original set
            # but also because the columns increase due to encoding
            if fold == 1:
                feature_names = list(X_train_final.columns)
                log(f"  > Features Used ({len(feature_names)}): {feature_names}")
            
            # -> Model Training 
            # params contains the hyperparameters for this configuration
            mlp_model = MLPRegressor(
                hidden_layer_sizes=params["hidden_layer_sizes"],
                activation=params.get("activation"),
                solver=params.get("solver"),
                learning_rate_init=params.get("learning_rate_init"),
                learning_rate=params.get("learning_rate"),
                batch_size=params.get("batch_size", "auto"),
                max_iter=params.get("max_iter"),
                n_iter_no_change=params.get("n_iter_no_change", 25),
                early_stopping=params.get("early_stopping"),
                random_state=42,
                verbose=False
            )
            mlp_model.fit(X_train_final, y_train)

            # predictions on train and validation fold
            y_pred_train = mlp_model.predict(X_train_final)
            y_pred_val   = mlp_model.predict(X_val_final)

            # -> Metrics Calculation
            # Training metrics:
            mae_tr = mean_absolute_error(y_train, y_pred_train)
            rmse_tr = np.sqrt(mean_squared_error(y_train, y_pred_train))
            r2_tr = r2_score(y_train, y_pred_train)
            bias_tr = np.mean(y_train - y_pred_train)
            # Validation metrics:
            mae_val = mean_absolute_error(y_val, y_pred_val)
            rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))
            r2_val = r2_score(y_val, y_pred_val)
            bias_val = np.mean(y_val - y_pred_val)
            med_ae_val = median_absolute_error(y_val, y_pred_val)

            # logging the metrics of this fold
            log(f"  > [TRAIN] R2: {r2_tr:.4f} | RMSE: {rmse_tr:.0f} | MAE: {mae_tr:.0f} | Bias: {bias_tr:.1f}")
            log(f"  > [VAL]   R2: {r2_val:.4f} | RMSE: {rmse_val:.0f} | MAE: {mae_val:.0f} | Bias: {bias_val:.1f}")
            # storing the metrics of this fold
            fold_maes_tr.append(mae_tr); fold_rmses_tr.append(rmse_tr); fold_r2s_tr.append(r2_tr); fold_bias_tr.append(bias_tr)
            fold_maes_val.append(mae_val); fold_rmses_val.append(rmse_val); fold_r2s_val.append(r2_val); fold_bias_val.append(bias_val)
            fold_med_ae.append(med_ae_val)

        # after all folds, we calculate the mean metrics across them
        mean_rmse_val = np.mean(fold_rmses_val)
        mean_mae_val  = np.mean(fold_maes_val)
        mean_r2_val   = np.mean(fold_r2s_val)
        mean_bias_val = np.mean(fold_bias_val)
        
        mean_mae_tr   = np.mean(fold_maes_tr)
        mean_r2_tr    = np.mean(fold_r2s_tr)
        mean_bias_tr  = np.mean(fold_bias_tr)
        
        # logging and storing these results
        log("")
        log(f"Config {config_id} SUMMARY:")
        log(f"  [TRAIN AVG] MAE: {mean_mae_tr:.1f} | R2: {mean_r2_tr:.4f} | Bias: {mean_bias_tr:.1f}")
        log(f"  [VAL AVG]   MAE: {mean_mae_val:.1f} | R2: {mean_r2_val:.4f} | Bias: {mean_bias_val:.1f} | RMSE: {mean_rmse_val:.1f}")

        search_results.append({
            "config_id": config_id,
            **params,
            
            "val_rmse": mean_rmse_val,
            "val_mae": mean_mae_val,
            "val_r2": mean_r2_val,
            "val_bias": mean_bias_val,

            "train_mae": mean_mae_tr,
            "train_r2": mean_r2_tr,
            "train_bias": mean_bias_tr,

            "val_med_ae": np.mean(fold_med_ae)
        })

        # our best configuration is defined by the best rmse
        # by best, we mean the lowest rmse
        if mean_rmse_val < best_rmse:
            best_rmse = mean_rmse_val
            best_config = {**params}
            log(f"[NEW BEST RMSE] Config {config_id}")

    log("# END OF SEARCH")


# Results
results_df = pd.DataFrame(search_results)
results_df_sorted = results_df.sort_values(by="val_rmse", ascending=True)

print("\nTop 5 Configurations by RMSE:")
display(results_df_sorted.head(10))

print("\nBest Config found:")
print(best_config)

# =============================
# START OF NN SEARCH (DETAILED METRICS: R2, RMSE, MAE, BIAS)
# LOG FILE SAVED TO: NN_2complete_log.txt
# =============================

######## CONFIG 1/10 ########
Parameters: {'solver': 'adam', 'n_iter_no_change': 25, 'max_iter': 900, 'learning_rate_init': 0.01, 'learning_rate': 'adaptive', 'hidden_layer_sizes': (128, 64), 'early_stopping': True, 'batch_size': 128, 'activation': 'relu'}

[C1|F1] Processing fold...
  > Features Used (16): ['year', 'mileage', 'engineSize', 'tax', 'mpg', 'previousOwners', 'Brand', 'model', 'transmission_AUTOMATIC', 'transmission_MANUAL', 'transmission_SEMIAUTO', 'fuelType_DIESEL', 'fuelType_ELECTRIC', 'fuelType_HYBRID', 'fuelType_OTHER', 'fuelType_PETROL']
  > [TRAIN] R2: 0.8634 | RMSE: 3616 | MAE: 2215 | Bias: 277.4
  > [VAL]   R2: 0.8695 | RMSE: 3448 | MAE: 2233 | Bias: 240.7

[C1|F2] Processing fold...
  > [TRAIN] R2: 0.8586 | RMSE: 3662 | MAE: 2203 | Bias: 427.5
  > [VAL]   R2: 0.8598 | RMSE: 3642 | MAE: 2203 | Bias:

,config_id,solver,n_iter_no_change,max_iter,learning_rate_init,learning_rate,hidden_layer_sizes,early_stopping,batch_size,activation,val_rmse,val_mae,val_r2,val_bias,train_mae,train_r2,train_bias,val_med_ae
7,8,adam,25,900,0.005,adaptive,"(128, 64)",True,128,relu,3541.975894,2196.098989,0.867635,-39.020119,2184.679394,0.871788,-40.720629,1503.110717
3,4,adam,25,900,0.005,adaptive,"(256, 128)",True,128,relu,3598.351673,2218.057170,0.863398,174.066072,2210.409403,0.866743,174.295965,1504.440089
5,6,adam,25,900,0.005,adaptive,"(256, 128)",True,64,relu,3599.308894,2215.313910,0.863335,66.791073,2208.660148,0.866267,66.192216,1499.075714
6,7,adam,25,900,0.010,adaptive,"(256, 128)",True,64,relu,3618.169043,2243.434702,0.861869,91.917749,2234.727105,0.864759,89.618981,1531.293802
4,5,adam,25,900,0.005,adaptive,"(128, 64)",True,64,relu,3653.295169,2248.879259,0.859191,35.625560,2240.756084,0.862221,37.065897,1537.058418
0,1,adam,25,900,0.010,adaptive,"(128, 64)",True,128,relu,3684.723544,2257.776083,0.856709,83.310565,2251.734854,0.860229,80.095036,1527.116432
9,10,adam,25,900,0.010,adaptive,"(256, 128)",True,128,relu,3707.659031,2293.999478,0.854913,150.910947,2288.850273,0.858356,148.915741,1582.234835
8,9,adam,25,900,0.010,adaptive,"(128, 64)",True,64,relu,3727.267410,2288.800603,0.853412,66.742078,2286.084242,0.855385,66.405242,1555.678347
1,2,adam,25,900,0.050,adaptive,"(128, 64)",True,128,relu,3766.921720,2316.402214,0.850309,-15.173122,2309.671891,0.853642,-17.687448,1596.360339
2,3,adam,25,900,0.050,adaptive,"(256, 128)",True,64,relu,3880.234772,2429.910958,0.840955,-166.171885,2425.056175,0.844482,-163.608522,1723.660633



Best Config found:
{'solver': 'adam', 'n_iter_no_change': 25, 'max_iter': 900, 'learning_rate_init': 0.005, 'learning_rate': 'adaptive', 'hidden_layer_sizes': (128, 64), 'early_stopping': True, 'batch_size': 128, 'activation': 'relu'}


### 3.2. Baseline Model using feature selection - opcional
<a id="baselinefs"></a>

### 3.3. Impact of Removing Previous Owners e log do preço no fs -2
<a id="noprevious"></a>

In [ ]:
# we will use an 5-fold cross-validation strategy
N_SPLITS = 5 
RANDOM_STATE = 42
# shuffles the data, in order to, at least theoretically, have more balanced folds
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
numeric_features = ["mileage", "engineSize", "tax", "mpg", "year"]
DROP_FROM_MODEL = ["previousOwners"]
# hyperparameter space that we are going to use to random search
# each configuration will get 1 value from each of these lists
param_distributions = {
    # MLP params
    "hidden_layer_sizes": [(256, 128),(128,64)],
    "learning_rate_init": [0.05, 0.01, 0.005],
    "activation": ["relu"],
    "solver": ["adam"],
    "batch_size": [64, 128],
    "max_iter": [900],
    "learning_rate": ["adaptive"],
    "early_stopping": [True],
    "n_iter_no_change": [25]
}

N_RANDOM_CONFIGS = 10 # this is a short random search, enough for baseline results

# we get N_RANDOM_CONFIGS (10) random parameter combinations from param_distributions 
sampler = ParameterSampler(param_distributions, n_iter=N_RANDOM_CONFIGS, random_state=RANDOM_STATE)

search_results = [] # to store results of each configuration, to store the ranking

# tracking the best configuration according to validation RMSE 
best_rmse = np.inf 
best_config = None 

# to store our fold results
log_path = "NN_noowners_logprice.txt"
# Note: this section is mainly for organizational purposes and is not particularly relevant 


with open(log_path, "w", encoding="utf-8") as log_file:

    # this function allows printing the messages to the screen and to the log file 
    def log(msg: str):
        print(msg) 
        log_file.write(msg + "\n") 
        log_file.flush() 

    # header for the log file - just for readability and general info
    log("# =============================")
    log(f"# START OF NN SEARCH (DETAILED METRICS: R2, RMSE, MAE, BIAS)")
    log(f"# LOG FILE SAVED TO: {log_path}")
    log("# =============================")
    
    # loop that iterates over each random configuration
    for config_id, params in enumerate(sampler, start=1):
        log("")
        log(f"######## CONFIG {config_id}/{N_RANDOM_CONFIGS} ########") 
        log(f"Parameters: {params}")

        # we are storing the metrics for each fold here
        fold_rmses_val, fold_maes_val, fold_r2s_val, fold_bias_val = [], [], [], []
        fold_rmses_tr, fold_maes_tr, fold_r2s_tr, fold_bias_tr = [], [], [], []
        fold_med_ae = []

        # for each configuration, we do K-Fold cross-validation
        for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
            
            # data split for the current fold
            X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
            y_train, y_val = y.iloc[train_idx].copy(), y.iloc[val_idx].copy()
            y_train_log = np.log1p(y_train)
            log(f"\n[C{config_id}|F{fold}] Processing fold...")


            # -> Preprocessing Steps 
            # fit is only done with the training data 
            # we aplly the transformations (some obtained through fit) to both train and validation sets
            # all preprocessing functions are explained before, here we just apply them in order
            year_state = fit_year_median(X_train, year_col="year", model_col="model")
            X_train = transform_year_with_model_median(X_train, state=year_state)
            X_val   = transform_year_with_model_median(X_val,   state=year_state)

            mileage_state = fit_mileage_imputer(X_train, "mileage", True)
            X_train = transform_mileage_imputer(X_train, mileage_state)
            X_val   = transform_mileage_imputer(X_val,   mileage_state)

            engine_state = fit_engine_size_imputer(X_train, engine_col="engineSize")
            X_train = transform_engine_size_imputer(X_train, state=engine_state)
            X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

            X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
            X_val   = transform_tax_custom_rules(X_val, "tax", "year", "fuelType", "engineSize")

            mpg_state = fit_mpg_imputer(X_train, mpg_col="mpg", do_abs=True)
            X_train = transform_mpg_imputer(X_train, state=mpg_state)
            X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

            brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
            X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
            X_val, _, _   = transform_ambiguous_brands(X_val, brand_state)

            model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
            X_train, _, _ = transform_invalid_models(X_train, model_state)
            X_val, _, _   = transform_invalid_models(X_val, model_state)

            transm_state = fit_transmission_resolver(X_train, valid_transmissions)
            X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
            X_val, _, _   = transform_transmission_resolver(X_val, transm_state)
            
            fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
            X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
            X_val, _, _   = transform_fueltype_resolver(X_val, fuel_state)

            X_train = X_train.drop(columns=DROP_FROM_MODEL, errors="ignore")
            X_val   = X_val.drop(columns=DROP_FROM_MODEL, errors="ignore")

            # -> Feature Encoding
            # applied to categorical features only
            # we separate high cardinality and low cardinality categorical features
            high_card_features = ["Brand", "model"] # target encoding for these
            low_card_features  = [c for c in categorical_features if c not in high_card_features]

            te = MyTargetEncoder(smoothing=5)
            te.fit(X_train[high_card_features], y_train_log) # fit on training data
            X_train_high = te.transform(X_train[high_card_features])
            X_val_high   = te.transform(X_val[high_card_features])

            ohe = MyOneHotEncoder()
            ohe.fit(X_train[low_card_features]) # fit on training data
            X_train_low = ohe.transform(X_train[low_card_features])
            X_val_low   = ohe.transform(X_val[low_card_features])

            # rejoining the categorical features (that are now encoded)
            X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
            X_val_cat   = pd.concat([X_val_high,   X_val_low],   axis=1)

            # joining numerical and categorical (encoded) features 
            X_train_final = pd.concat([X_train[numeric_features], X_train_cat], axis=1)
            X_val_final   = pd.concat([X_val[numeric_features],   X_val_cat],   axis=1)

            
            # here, we log the features name, to keep track of what is being used from the original set
            # but also because the columns increase due to encoding
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train_final)
            X_val_scaled   = scaler.transform(X_val_final)

            if fold == 1:
                feature_names = list(X_train_final.columns)
                log(f"  > Features Used ({len(feature_names)}): {feature_names}")

            # -> Model Training 
            # params contains the hyperparameters for this configuration
            mlp_model = MLPRegressor(
                hidden_layer_sizes=params["hidden_layer_sizes"],
                activation=params.get("activation"),
                solver=params.get("solver"),
                learning_rate_init=params.get("learning_rate_init"),
                learning_rate=params.get("learning_rate"),
                batch_size=params.get("batch_size", "auto"),
                max_iter=params.get("max_iter"),
                n_iter_no_change=params.get("n_iter_no_change", 25),
                early_stopping=params.get("early_stopping"),
                random_state=42,
                verbose=False
            )
            mlp_model.fit(X_train_scaled, y_train_log)

            # predictions on train and validation fold
            y_pred_train_log = mlp_model.predict(X_train_scaled)
            y_pred_val_log   = mlp_model.predict(X_val_scaled)

            y_pred_train = np.expm1(y_pred_train_log)
            y_pred_val   = np.expm1(y_pred_val_log)

            # -> Metrics Calculation
            # Training metrics:
            mae_tr = mean_absolute_error(y_train, y_pred_train)
            rmse_tr = np.sqrt(mean_squared_error(y_train, y_pred_train))
            r2_tr = r2_score(y_train, y_pred_train)
            bias_tr = np.mean(y_train - y_pred_train)
            # Validation metrics:
            mae_val = mean_absolute_error(y_val, y_pred_val)
            rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))
            r2_val = r2_score(y_val, y_pred_val)
            bias_val = np.mean(y_val - y_pred_val)
            med_ae_val = median_absolute_error(y_val, y_pred_val)

            # logging the metrics of this fold
            log(f"  > [TRAIN] R2: {r2_tr:.4f} | RMSE: {rmse_tr:.0f} | MAE: {mae_tr:.0f} | Bias: {bias_tr:.1f}")
            log(f"  > [VAL]   R2: {r2_val:.4f} | RMSE: {rmse_val:.0f} | MAE: {mae_val:.0f} | Bias: {bias_val:.1f}")
            # storing the metrics of this fold
            fold_maes_tr.append(mae_tr); fold_rmses_tr.append(rmse_tr); fold_r2s_tr.append(r2_tr); fold_bias_tr.append(bias_tr)
            fold_maes_val.append(mae_val); fold_rmses_val.append(rmse_val); fold_r2s_val.append(r2_val); fold_bias_val.append(bias_val)
            fold_med_ae.append(med_ae_val)

        # after all folds, we calculate the mean metrics across them
        mean_rmse_val = np.mean(fold_rmses_val)
        mean_mae_val  = np.mean(fold_maes_val)
        mean_r2_val   = np.mean(fold_r2s_val)
        mean_bias_val = np.mean(fold_bias_val)
        
        mean_mae_tr   = np.mean(fold_maes_tr)
        mean_r2_tr    = np.mean(fold_r2s_tr)
        mean_bias_tr  = np.mean(fold_bias_tr)
        
        # logging and storing these results
        log("")
        log(f"Config {config_id} SUMMARY:")
        log(f"  [TRAIN AVG] MAE: {mean_mae_tr:.1f} | R2: {mean_r2_tr:.4f} | Bias: {mean_bias_tr:.1f}")
        log(f"  [VAL AVG]   MAE: {mean_mae_val:.1f} | R2: {mean_r2_val:.4f} | Bias: {mean_bias_val:.1f} | RMSE: {mean_rmse_val:.1f}")

        search_results.append({
            "config_id": config_id,
            **params,
            
            "val_rmse": mean_rmse_val,
            "val_mae": mean_mae_val,
            "val_r2": mean_r2_val,
            "val_bias": mean_bias_val,

            "train_mae": mean_mae_tr,
            "train_r2": mean_r2_tr,
            "train_bias": mean_bias_tr,

            "val_med_ae": np.mean(fold_med_ae)
        })

        # our best configuration is defined by the best rmse
        # by best, we mean the lowest rmse
        if mean_rmse_val < best_rmse:
            best_rmse = mean_rmse_val
            best_config = {**params}
            log(f"[NEW BEST RMSE] Config {config_id}")

    log("# END OF SEARCH")


# Results
results_df = pd.DataFrame(search_results)
results_df_sorted = results_df.sort_values(by="val_rmse", ascending=True)

print("\nTop 5 Configurations by RMSE:")
display(results_df_sorted.head(10))

print("\nBest Config found:")
print(best_config)

# =============================
# START OF NN SEARCH (DETAILED METRICS: R2, RMSE, MAE, BIAS)
# LOG FILE SAVED TO: NN_noowners_logprice.txt
# =============================

######## CONFIG 1/10 ########
Parameters: {'solver': 'adam', 'n_iter_no_change': 25, 'max_iter': 900, 'learning_rate_init': 0.01, 'learning_rate': 'adaptive', 'hidden_layer_sizes': (128, 64), 'early_stopping': True, 'batch_size': 128, 'activation': 'relu'}

[C1|F1] Processing fold...
  > Features Used (15): ['mileage', 'engineSize', 'tax', 'mpg', 'year', 'Brand', 'model', 'transmission_AUTOMATIC', 'transmission_MANUAL', 'transmission_SEMIAUTO', 'fuelType_DIESEL', 'fuelType_ELECTRIC', 'fuelType_HYBRID', 'fuelType_OTHER', 'fuelType_PETROL']
  > [TRAIN] R2: 0.9141 | RMSE: 2867 | MAE: 1712 | Bias: 79.8
  > [VAL]   R2: 0.9166 | RMSE: 2755 | MAE: 1730 | Bias: 33.8

[C1|F2] Processing fold...
  > [TRAIN] R2: 0.9106 | RMSE: 2912 | MAE: 1707 | Bias: 119.2
  > [VAL]   R2: 0.9053 | RMSE: 2993 | MAE: 1753 | Bias: 139.2

[C1|F3] 

,config_id,solver,n_iter_no_change,max_iter,learning_rate_init,learning_rate,hidden_layer_sizes,early_stopping,batch_size,activation,val_rmse,val_mae,val_r2,val_bias,train_mae,train_r2,train_bias,val_med_ae
5,6,adam,25,900,0.005,adaptive,"(256, 128)",True,64,relu,2.955609e+03,1748.665946,9.077264e-01,-36.420291,1718.875297,9.123759e-01,-42.238033,1126.215791
0,1,adam,25,900,0.010,adaptive,"(128, 64)",True,128,relu,2.990074e+03,1758.130099,9.055593e-01,27.654105,1719.427977,9.127058e-01,25.130324,1119.766822
4,5,adam,25,900,0.005,adaptive,"(128, 64)",True,64,relu,3.022066e+03,1785.436068,9.034372e-01,41.589732,1751.934563,9.088124e-01,37.637949,1127.121193
6,7,adam,25,900,0.010,adaptive,"(256, 128)",True,64,relu,3.034206e+03,1783.536947,9.028115e-01,88.244625,1751.221860,9.094600e-01,78.150063,1130.481610
3,4,adam,25,900,0.005,adaptive,"(256, 128)",True,128,relu,3.064994e+03,1789.782465,9.006964e-01,315.919010,1759.685018,9.057782e-01,317.356445,1126.639536
7,8,adam,25,900,0.005,adaptive,"(128, 64)",True,128,relu,3.074014e+03,1722.966561,8.998630e-01,182.895664,1680.268946,9.126809e-01,176.345930,1079.450972
9,10,adam,25,900,0.010,adaptive,"(256, 128)",True,128,relu,3.577240e+03,1806.815598,8.481795e-01,99.104081,1774.120240,8.763857e-01,95.178913,1102.876655
1,2,adam,25,900,0.050,adaptive,"(128, 64)",True,128,relu,3.932983e+03,2225.192939,8.257119e-01,151.017798,2205.765916,7.768405e-01,152.324136,1443.692238
2,3,adam,25,900,0.050,adaptive,"(256, 128)",True,64,relu,4.845087e+03,3011.155283,6.820970e-01,861.947340,2984.988892,6.853766e-01,851.557880,2037.717130
8,9,adam,25,900,0.010,adaptive,"(128, 64)",True,64,relu,6.640901e+06,55638.197243,-2.182925e+06,-53706.847242,44568.036233,-2.264859e+06,-42668.026365,1120.471057



Best Config found:
{'solver': 'adam', 'n_iter_no_change': 25, 'max_iter': 900, 'learning_rate_init': 0.005, 'learning_rate': 'adaptive', 'hidden_layer_sizes': (256, 128), 'early_stopping': True, 'batch_size': 64, 'activation': 'relu'}


###  3.5. Model Performance with Log-Target, Age Feature, without Feature Selection (No Previous Owners) - 3
<a id="log_age"></a>

In [ ]:
# we will use an 5-fold cross-validation strategy
N_SPLITS = 5 
RANDOM_STATE = 42
# shuffles the data, in order to, at least theoretically, have more balanced folds
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
numeric_features = ["mileage", "engineSize", "tax", "mpg", "age"]
DROP_FROM_MODEL = ["previousOwners"]
# hyperparameter space that we are going to use to random search
# each configuration will get 1 value from each of these lists
param_distributions = {
    # MLP params
    "hidden_layer_sizes": [(256, 128),(128,64)],
    "learning_rate_init": [0.05, 0.01, 0.005],
    "activation": ["relu"],
    "solver": ["adam"],
    "batch_size": [64, 128],
    "max_iter": [900],
    "learning_rate": ["adaptive"],
    "early_stopping": [True],
    "n_iter_no_change": [25]
}

N_RANDOM_CONFIGS = 10 # this is a short random search, enough for baseline results

# we get N_RANDOM_CONFIGS (10) random parameter combinations from param_distributions 
sampler = ParameterSampler(param_distributions, n_iter=N_RANDOM_CONFIGS, random_state=RANDOM_STATE)

search_results = [] # to store results of each configuration, to store the ranking

# tracking the best configuration according to validation RMSE 
best_rmse = np.inf 
best_config = None 

# to store our fold results
log_path = "NN_noowners_age_logprice.txt"
# Note: this section is mainly for organizational purposes and is not particularly relevant 


with open(log_path, "w", encoding="utf-8") as log_file:

    # this function allows printing the messages to the screen and to the log file 
    def log(msg: str):
        print(msg) 
        log_file.write(msg + "\n") 
        log_file.flush() 

    # header for the log file - just for readability and general info
    log("# =============================")
    log(f"# START OF NN SEARCH (DETAILED METRICS: R2, RMSE, MAE, BIAS)")
    log(f"# LOG FILE SAVED TO: {log_path}")
    log("# =============================")
    
    # loop that iterates over each random configuration
    for config_id, params in enumerate(sampler, start=1):
        log("")
        log(f"######## CONFIG {config_id}/{N_RANDOM_CONFIGS} ########") 
        log(f"Parameters: {params}")

        # we are storing the metrics for each fold here
        fold_rmses_val, fold_maes_val, fold_r2s_val, fold_bias_val = [], [], [], []
        fold_rmses_tr, fold_maes_tr, fold_r2s_tr, fold_bias_tr = [], [], [], []
        fold_med_ae = []

        # for each configuration, we do K-Fold cross-validation
        for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
            
            # data split for the current fold
            X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
            y_train, y_val = y.iloc[train_idx].copy(), y.iloc[val_idx].copy()
            y_train_log = np.log1p(y_train)
            log(f"\n[C{config_id}|F{fold}] Processing fold...")


            # -> Preprocessing Steps 
            # fit is only done with the training data 
            # we aplly the transformations (some obtained through fit) to both train and validation sets
            # all preprocessing functions are explained before, here we just apply them in order
            year_state = fit_year_median(X_train, year_col="year", model_col="model")
            X_train = transform_year_with_model_median(X_train, state=year_state)
            X_val   = transform_year_with_model_median(X_val,   state=year_state)

            mileage_state = fit_mileage_imputer(X_train, "mileage", True)
            X_train = transform_mileage_imputer(X_train, mileage_state)
            X_val   = transform_mileage_imputer(X_val,   mileage_state)

            engine_state = fit_engine_size_imputer(X_train, engine_col="engineSize")
            X_train = transform_engine_size_imputer(X_train, state=engine_state)
            X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

            X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
            X_val   = transform_tax_custom_rules(X_val, "tax", "year", "fuelType", "engineSize")

            mpg_state = fit_mpg_imputer(X_train, mpg_col="mpg", do_abs=True)
            X_train = transform_mpg_imputer(X_train, state=mpg_state)
            X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

            brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
            X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
            X_val, _, _   = transform_ambiguous_brands(X_val, brand_state)

            model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
            X_train, _, _ = transform_invalid_models(X_train, model_state)
            X_val, _, _   = transform_invalid_models(X_val, model_state)

            transm_state = fit_transmission_resolver(X_train, valid_transmissions)
            X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
            X_val, _, _   = transform_transmission_resolver(X_val, transm_state)
            
            fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
            X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
            X_val, _, _   = transform_fueltype_resolver(X_val, fuel_state)

             # -> Feature engineering: create age and drop year (do this after all year-based steps)
            X_train = create_age_and_drop_year(X_train, year_col="year", base_year=2020)
            X_val   = create_age_and_drop_year(X_val,   year_col="year", base_year=2020)

            # hard guarantee: previousOwners is not used by the model
            X_train = X_train.drop(columns=DROP_FROM_MODEL, errors="ignore")
            X_val   = X_val.drop(columns=DROP_FROM_MODEL, errors="ignore")

            # -> Feature Encoding
            # applied to categorical features only
            # we separate high cardinality and low cardinality categorical features
            high_card_features = ["Brand", "model"] # target encoding for these
            low_card_features  = [c for c in categorical_features if c not in high_card_features]

            te = MyTargetEncoder(smoothing=5)
            te.fit(X_train[high_card_features], y_train_log) # fit on training data
            X_train_high = te.transform(X_train[high_card_features])
            X_val_high   = te.transform(X_val[high_card_features])

            ohe = MyOneHotEncoder()
            ohe.fit(X_train[low_card_features]) # fit on training data
            X_train_low = ohe.transform(X_train[low_card_features])
            X_val_low   = ohe.transform(X_val[low_card_features])

            # rejoining the categorical features (that are now encoded)
            X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
            X_val_cat   = pd.concat([X_val_high,   X_val_low],   axis=1)

            # joining numerical and categorical (encoded) features 
            X_train_final = pd.concat([X_train[numeric_features], X_train_cat], axis=1)
            X_val_final   = pd.concat([X_val[numeric_features],   X_val_cat],   axis=1)

            
            # here, we log the features name, to keep track of what is being used from the original set
            # but also because the columns increase due to encoding
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train_final)
            X_val_scaled   = scaler.transform(X_val_final)

            # align columns to avoid mismatch due to encoding
            X_val_final = X_val_final.reindex(columns=X_train_final.columns, fill_value=0)

            if fold == 1:
                feature_names = list(X_train_final.columns)
                log(f"  > Features Used ({len(feature_names)}): {feature_names}")

            # -> Model Training 
            # params contains the hyperparameters for this configuration
            mlp_model = MLPRegressor(
                hidden_layer_sizes=params["hidden_layer_sizes"],
                activation=params.get("activation"),
                solver=params.get("solver"),
                learning_rate_init=params.get("learning_rate_init"),
                learning_rate=params.get("learning_rate"),
                batch_size=params.get("batch_size", "auto"),
                max_iter=params.get("max_iter"),
                n_iter_no_change=params.get("n_iter_no_change", 25),
                early_stopping=params.get("early_stopping"),
                random_state=42,
                verbose=False
            )
            mlp_model.fit(X_train_scaled, y_train_log)

            # predictions on train and validation fold
            y_pred_train_log = mlp_model.predict(X_train_scaled)
            y_pred_val_log   = mlp_model.predict(X_val_scaled)

            y_pred_train = np.expm1(y_pred_train_log)
            y_pred_val   = np.expm1(y_pred_val_log)

            # -> Metrics Calculation
            # Training metrics:
            mae_tr = mean_absolute_error(y_train, y_pred_train)
            rmse_tr = np.sqrt(mean_squared_error(y_train, y_pred_train))
            r2_tr = r2_score(y_train, y_pred_train)
            bias_tr = np.mean(y_train - y_pred_train)
            # Validation metrics:
            mae_val = mean_absolute_error(y_val, y_pred_val)
            rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))
            r2_val = r2_score(y_val, y_pred_val)
            bias_val = np.mean(y_val - y_pred_val)
            med_ae_val = median_absolute_error(y_val, y_pred_val)

            # logging the metrics of this fold
            log(f"  > [TRAIN] R2: {r2_tr:.4f} | RMSE: {rmse_tr:.0f} | MAE: {mae_tr:.0f} | Bias: {bias_tr:.1f}")
            log(f"  > [VAL]   R2: {r2_val:.4f} | RMSE: {rmse_val:.0f} | MAE: {mae_val:.0f} | Bias: {bias_val:.1f}")
            # storing the metrics of this fold
            fold_maes_tr.append(mae_tr); fold_rmses_tr.append(rmse_tr); fold_r2s_tr.append(r2_tr); fold_bias_tr.append(bias_tr)
            fold_maes_val.append(mae_val); fold_rmses_val.append(rmse_val); fold_r2s_val.append(r2_val); fold_bias_val.append(bias_val)
            fold_med_ae.append(med_ae_val)

        # after all folds, we calculate the mean metrics across them
        mean_rmse_val = np.mean(fold_rmses_val)
        mean_mae_val  = np.mean(fold_maes_val)
        mean_r2_val   = np.mean(fold_r2s_val)
        mean_bias_val = np.mean(fold_bias_val)
        
        mean_mae_tr   = np.mean(fold_maes_tr)
        mean_r2_tr    = np.mean(fold_r2s_tr)
        mean_bias_tr  = np.mean(fold_bias_tr)
        
        # logging and storing these results
        log("")
        log(f"Config {config_id} SUMMARY:")
        log(f"  [TRAIN AVG] MAE: {mean_mae_tr:.1f} | R2: {mean_r2_tr:.4f} | Bias: {mean_bias_tr:.1f}")
        log(f"  [VAL AVG]   MAE: {mean_mae_val:.1f} | R2: {mean_r2_val:.4f} | Bias: {mean_bias_val:.1f} | RMSE: {mean_rmse_val:.1f}")

        search_results.append({
            "config_id": config_id,
            **params,
            
            "val_rmse": mean_rmse_val,
            "val_mae": mean_mae_val,
            "val_r2": mean_r2_val,
            "val_bias": mean_bias_val,

            "train_mae": mean_mae_tr,
            "train_r2": mean_r2_tr,
            "train_bias": mean_bias_tr,

            "val_med_ae": np.mean(fold_med_ae)
        })

        # our best configuration is defined by the best rmse
        # by best, we mean the lowest rmse
        if mean_rmse_val < best_rmse:
            best_rmse = mean_rmse_val
            best_config = {**params}
            log(f"[NEW BEST RMSE] Config {config_id}")

    log("# END OF SEARCH")


# Results
results_df = pd.DataFrame(search_results)
results_df_sorted = results_df.sort_values(by="val_rmse", ascending=True)

print("\nTop 5 Configurations by RMSE:")
display(results_df_sorted.head(10))

print("\nBest Config found:")
print(best_config)

# =============================
# START OF NN SEARCH (DETAILED METRICS: R2, RMSE, MAE, BIAS)
# LOG FILE SAVED TO: NN_noowners_age_logprice.txt
# =============================

######## CONFIG 1/10 ########
Parameters: {'solver': 'adam', 'n_iter_no_change': 25, 'max_iter': 900, 'learning_rate_init': 0.01, 'learning_rate': 'adaptive', 'hidden_layer_sizes': (128, 64), 'early_stopping': True, 'batch_size': 128, 'activation': 'relu'}

[C1|F1] Processing fold...
  > Features Used (15): ['mileage', 'engineSize', 'tax', 'mpg', 'age', 'Brand', 'model', 'transmission_AUTOMATIC', 'transmission_MANUAL', 'transmission_SEMIAUTO', 'fuelType_DIESEL', 'fuelType_ELECTRIC', 'fuelType_HYBRID', 'fuelType_OTHER', 'fuelType_PETROL']
  > [TRAIN] R2: 0.9139 | RMSE: 2871 | MAE: 1713 | Bias: 404.8
  > [VAL]   R2: 0.9163 | RMSE: 2762 | MAE: 1743 | Bias: 362.2

[C1|F2] Processing fold...
  > [TRAIN] R2: 0.9058 | RMSE: 2990 | MAE: 1745 | Bias: 348.6
  > [VAL]   R2: 0.9014 | RMSE: 3054 | MAE: 1786 | Bias: 378.2

[C1

,config_id,solver,n_iter_no_change,max_iter,learning_rate_init,learning_rate,hidden_layer_sizes,early_stopping,batch_size,activation,val_rmse,val_mae,val_r2,val_bias,train_mae,train_r2,train_bias,val_med_ae
7,8,adam,25,900,0.005,adaptive,"(128, 64)",True,128,relu,2928.273188,1716.228911,0.909464,39.355923,1673.107657,0.916802,31.946904,1089.886092
9,10,adam,25,900,0.010,adaptive,"(256, 128)",True,128,relu,2994.776836,1729.015973,0.905280,368.232619,1691.439998,0.912918,367.702438,1078.196528
5,6,adam,25,900,0.005,adaptive,"(256, 128)",True,64,relu,3007.872502,1746.999522,0.904503,235.739317,1719.372751,0.909701,231.236865,1105.059387
6,7,adam,25,900,0.010,adaptive,"(256, 128)",True,64,relu,3027.936760,1779.526294,0.903207,278.616410,1754.374909,0.908253,268.639172,1109.531912
8,9,adam,25,900,0.010,adaptive,"(128, 64)",True,64,relu,3052.360412,1783.887459,0.901541,207.956691,1755.932637,0.907709,209.571707,1123.133106
0,1,adam,25,900,0.010,adaptive,"(128, 64)",True,128,relu,3087.820937,1788.078115,0.899215,357.557668,1757.879815,0.906467,355.448900,1123.953734
3,4,adam,25,900,0.005,adaptive,"(256, 128)",True,128,relu,3248.124270,1781.338499,0.887231,116.850248,1742.701962,0.902604,112.667304,1133.447822
1,2,adam,25,900,0.050,adaptive,"(128, 64)",True,128,relu,3259.342532,1914.091665,0.887779,510.874370,1896.834295,0.892251,498.051456,1211.406965
4,5,adam,25,900,0.005,adaptive,"(128, 64)",True,64,relu,3453.640048,1770.935085,0.864431,-11.743333,1733.924881,0.878490,-19.942061,1116.604309
2,3,adam,25,900,0.050,adaptive,"(256, 128)",True,64,relu,6150.974097,3984.462806,0.499367,975.064051,3943.689567,0.503156,946.017394,2800.029722



Best Config found:
{'solver': 'adam', 'n_iter_no_change': 25, 'max_iter': 900, 'learning_rate_init': 0.005, 'learning_rate': 'adaptive', 'hidden_layer_sizes': (128, 64), 'early_stopping': True, 'batch_size': 128, 'activation': 'relu'}


###  3.5. Model Performance with Log-Target, Age Feature, with Feature Selection (No Previous Owners) - 4

In [ ]:
# numeric features used by the model (year is replaced by age; previousOwners is excluded)
numeric_features = ["mileage", "engineSize", "tax", "mpg", "age"]



# shuffles the data, in order to (at least theoretically) have more balanced folds
# we will use an 8-fold cross-validation strategy
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# we explicitly exclude this feature from the model input
DROP_FROM_MODEL = ["previousOwners"]

# feature selection: keep 90% of the final encoded feature space
FS_KEEP_RATIO = 0.80

# RandomForest used only to rank feature importances for SelectFromModel
RF_FS_PARAMS = {
    "n_estimators": 500,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "bootstrap": True,
}

# hyperparameter space that we are going to use for random search
# each configuration will get 1 value from each of these lists
param_distributions = {
    # MLP params
    "hidden_layer_sizes": [(256, 128),(128,64)],
    "learning_rate_init": [0.05, 0.01, 0.005],
    "activation": ["relu"],
    "solver": ["adam"],
    "batch_size": [64, 128],
    "max_iter": [900],
    "learning_rate": ["adaptive"],
    "early_stopping": [True],
    "n_iter_no_change": [25]
}


N_RANDOM_CONFIGS = 15  # short random search, enough for baseline results

# we get N_RANDOM_CONFIGS random parameter combinations from param_distributions
sampler = ParameterSampler(param_distributions, n_iter=N_RANDOM_CONFIGS, random_state=RANDOM_STATE)

search_results = []  # stores results for ranking
best_rmse = np.inf
best_config = None

# log setup (mainly for development and traceability)
log_path = "NN_noowners_age_log_fs.txt"

# Note: this section is included mainly for organizational purposes
with open(log_path, "w", encoding="utf-8") as log_file:

    def log(msg: str):
        print(msg)
        log_file.write(msg + "\n")
        log_file.flush()

    log("# =============================")
    log("# START OF NN SEARCH (LOG TARGET + AGE FEATURE + FS 80%)")
    log("# METRICS ON ORIGINAL SCALE: R2, RMSE, MAE, BIAS")
    log(f"# LOG FILE SAVED TO: {log_path}")
    log("# =============================")

    for config_id, params in enumerate(sampler, start=1):
        log("")
        log(f"######## CONFIG {config_id}/{N_RANDOM_CONFIGS} ########")
        log(f"Parameters: {params}")

        fold_rmses_val, fold_maes_val, fold_r2s_val, fold_bias_val = [], [], [], []
        fold_rmses_tr,  fold_maes_tr,  fold_r2s_tr,  fold_bias_tr  = [], [], [], []
        fold_med_ae = []
        fold_nsel = []

        for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):

            X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
            y_train, y_val = y.iloc[train_idx].copy(), y.iloc[val_idx].copy()

            # log-transform target for training
            y_train_log = np.log1p(y_train)

            log(f"\n[C{config_id}|F{fold}] Processing fold...")

            # -> Preprocessing steps (fit only on training fold, apply to both)
            year_state = fit_year_median(X_train, year_col="year", model_col="model")
            X_train = transform_year_with_model_median(X_train, state=year_state)
            X_val   = transform_year_with_model_median(X_val,   state=year_state)

            mileage_state = fit_mileage_imputer(X_train, mileage_col="mileage", do_abs=True)
            X_train = transform_mileage_imputer(X_train, state=mileage_state)
            X_val   = transform_mileage_imputer(X_val,   state=mileage_state)

            engine_state = fit_engine_size_imputer(X_train, engine_col="engineSize")
            X_train = transform_engine_size_imputer(X_train, state=engine_state)
            X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

            X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
            X_val   = transform_tax_custom_rules(X_val,   "tax", "year", "fuelType", "engineSize")

            mpg_state = fit_mpg_imputer(X_train, mpg_col="mpg", do_abs=True)
            X_train = transform_mpg_imputer(X_train, state=mpg_state)
            X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

            brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
            X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
            X_val,   _, _ = transform_ambiguous_brands(X_val,   brand_state)

            model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
            X_train, _, _ = transform_invalid_models(X_train, model_state)
            X_val,   _, _ = transform_invalid_models(X_val,   model_state)

            transm_state = fit_transmission_resolver(X_train, valid_transmissions)
            X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
            X_val,   _, _ = transform_transmission_resolver(X_val,   transm_state)

            fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
            X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
            X_val,   _, _ = transform_fueltype_resolver(X_val,   fuel_state)

            # -> Feature engineering: create age and drop year (do this after all year-based steps)
            X_train = create_age_and_drop_year(X_train, year_col="year", base_year=2020)
            X_val   = create_age_and_drop_year(X_val,   year_col="year", base_year=2020)

            # hard guarantee: previousOwners is not used by the model
            X_train = X_train.drop(columns=DROP_FROM_MODEL, errors="ignore")
            X_val   = X_val.drop(columns=DROP_FROM_MODEL, errors="ignore")

            # -> Feature encoding (categorical only)
            high_card_features = ["Brand", "model"]
            low_card_features  = [c for c in categorical_features if c not in high_card_features]

            te = MyTargetEncoder(smoothing=5)
            te.fit(X_train[high_card_features], y_train_log)  # fit using log-target
            X_train_high = te.transform(X_train[high_card_features])
            X_val_high   = te.transform(X_val[high_card_features])

            ohe = MyOneHotEncoder()
            ohe.fit(X_train[low_card_features])
            X_train_low = ohe.transform(X_train[low_card_features])
            X_val_low   = ohe.transform(X_val[low_card_features])

            X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
            X_val_cat   = pd.concat([X_val_high,   X_val_low],   axis=1)

            # join numerical and categorical (encoded) features
            X_train_final = pd.concat([X_train[numeric_features], X_train_cat], axis=1)
            X_val_final   = pd.concat([X_val[numeric_features],   X_val_cat],   axis=1)

            if len(numeric_features) > 0:
                scaler = StandardScaler()
                X_train_final[numeric_features] = scaler.fit_transform(X_train_final[numeric_features])
                X_val_final[numeric_features]   = scaler.transform(X_val_final[numeric_features])

            # align columns to avoid mismatch due to encoding
            X_val_final = X_val_final.reindex(columns=X_train_final.columns, fill_value=0)

            if fold == 1:
                feature_names = list(X_train_final.columns)
                log(f"  > Features before FS ({len(feature_names)}): {feature_names}")

            # -> Feature selection (keep top 90% features by RF importance)
            n_feats = X_train_final.shape[1]
            k = int(np.ceil(FS_KEEP_RATIO * n_feats))
            k = max(1, min(k, n_feats))

            rf_fs = RandomForestRegressor(**RF_FS_PARAMS)
            rf_fs.fit(X_train_final, y_train_log)

            selector = SelectFromModel(
                estimator=rf_fs,
                threshold=-np.inf,  # keep the top max_features
                max_features=k,
                prefit=True
            )

            selected_cols = X_train_final.columns[selector.get_support()]
            fold_nsel.append(len(selected_cols))

            X_train_sel = X_train_final[selected_cols]
            X_val_sel   = X_val_final[selected_cols]

            if fold == 1:
                log(f"  > Selected features ({len(selected_cols)}/{n_feats}) with FS_KEEP_RATIO={FS_KEEP_RATIO:.2f}")

            # -> Model training (train on log-target)
            mlp_model = MLPRegressor(
                hidden_layer_sizes=params["hidden_layer_sizes"],
                activation=params.get("activation"),
                solver=params.get("solver"),
                learning_rate_init=params.get("learning_rate_init"),
                learning_rate=params.get("learning_rate"),
                batch_size=params.get("batch_size", "auto"),
                max_iter=params.get("max_iter"),
                n_iter_no_change=params.get("n_iter_no_change", 25),
                early_stopping=params.get("early_stopping"),
                random_state=42,
                verbose=False
            )
            mlp_model.fit(X_train_sel, y_train_log)

            # predictions (log scale), then invert to original scale
            y_pred_train_log = mlp_model.predict(X_train_sel)
            y_pred_val_log   = mlp_model.predict(X_val_sel)

            y_pred_train = np.expm1(y_pred_train_log)
            y_pred_val   = np.expm1(y_pred_val_log)

            # -> Metrics on original scale
            mae_tr  = mean_absolute_error(y_train, y_pred_train)
            rmse_tr = np.sqrt(mean_squared_error(y_train, y_pred_train))
            r2_tr   = r2_score(y_train, y_pred_train)
            bias_tr = np.mean(y_train - y_pred_train)

            mae_val  = mean_absolute_error(y_val, y_pred_val)
            rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))
            r2_val   = r2_score(y_val, y_pred_val)
            bias_val = np.mean(y_val - y_pred_val)

            med_ae_val = median_absolute_error(y_val, y_pred_val)

            log(f"  > [TRAIN] R2: {r2_tr:.4f} | RMSE: {rmse_tr:.0f} | MAE: {mae_tr:.0f} | Bias: {bias_tr:.1f}")
            log(f"  > [VAL]   R2: {r2_val:.4f} | RMSE: {rmse_val:.0f} | MAE: {mae_val:.0f} | Bias: {bias_val:.1f}")

            fold_maes_tr.append(mae_tr);   fold_rmses_tr.append(rmse_tr);   fold_r2s_tr.append(r2_tr);   fold_bias_tr.append(bias_tr)
            fold_maes_val.append(mae_val); fold_rmses_val.append(rmse_val); fold_r2s_val.append(r2_val); fold_bias_val.append(bias_val)
            fold_med_ae.append(med_ae_val)

        mean_rmse_val = np.mean(fold_rmses_val)
        mean_mae_val  = np.mean(fold_maes_val)
        mean_r2_val   = np.mean(fold_r2s_val)
        mean_bias_val = np.mean(fold_bias_val)

        mean_mae_tr   = np.mean(fold_maes_tr)
        mean_r2_tr    = np.mean(fold_r2s_tr)
        mean_bias_tr  = np.mean(fold_bias_tr)

        mean_nsel = float(np.mean(fold_nsel))

        log("")
        log(f"Config {config_id} SUMMARY:")
        log(f"  Selected features (avg): {mean_nsel:.0f} / {n_feats} (after encoding)")
        log(f"  [TRAIN AVG] MAE: {mean_mae_tr:.1f} | R2: {mean_r2_tr:.4f} | Bias: {mean_bias_tr:.1f}")
        log(f"  [VAL AVG]   MAE: {mean_mae_val:.1f} | R2: {mean_r2_val:.4f} | Bias: {mean_bias_val:.1f} | RMSE: {mean_rmse_val:.1f}")

        search_results.append({
            "config_id": config_id,
            **params,

            "fs_keep_ratio": FS_KEEP_RATIO,
            "avg_selected_features": mean_nsel,

            "val_rmse": mean_rmse_val,
            "val_mae": mean_mae_val,
            "val_r2": mean_r2_val,
            "val_bias": mean_bias_val,

            "train_mae": mean_mae_tr,
            "train_r2": mean_r2_tr,
            "train_bias": mean_bias_tr,

            "val_med_ae": np.mean(fold_med_ae)
        })

        if mean_rmse_val < best_rmse:
            best_rmse = mean_rmse_val
            best_config = {**params}
            log(f"[NEW BEST RMSE] Config {config_id}")

    log("# END OF SEARCH")

# Results
results_df = pd.DataFrame(search_results)
results_df_sorted = results_df.sort_values(by="val_rmse", ascending=True)

print("\nTop 5 Configurations by RMSE:")
display(results_df_sorted.head(5))

print("\nBest Config found:")
print(best_config)


# =============================
# START OF NN SEARCH (LOG TARGET + AGE FEATURE + FS 80%)
# METRICS ON ORIGINAL SCALE: R2, RMSE, MAE, BIAS
# LOG FILE SAVED TO: NN_noowners_age_log_fs.txt
# =============================

######## CONFIG 1/15 ########
Parameters: {'solver': 'adam', 'n_iter_no_change': 25, 'max_iter': 900, 'learning_rate_init': 0.05, 'learning_rate': 'adaptive', 'hidden_layer_sizes': (256, 128), 'early_stopping': True, 'batch_size': 64, 'activation': 'relu'}

[C1|F1] Processing fold...


/Users/franciscafernandes/anaconda3/lib/python3.13/site-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 12 is smaller than n_iter=15. Running 12 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


  > Features before FS (15): ['mileage', 'engineSize', 'tax', 'mpg', 'age', 'Brand', 'model', 'transmission_AUTOMATIC', 'transmission_MANUAL', 'transmission_SEMIAUTO', 'fuelType_DIESEL', 'fuelType_ELECTRIC', 'fuelType_HYBRID', 'fuelType_OTHER', 'fuelType_PETROL']
  > Selected features (12/15) with FS_KEEP_RATIO=0.80
  > [TRAIN] R2: 0.8700 | RMSE: 3522 | MAE: 2101 | Bias: 506.6
  > [VAL]   R2: 0.8754 | RMSE: 3358 | MAE: 2105 | Bias: 482.0

[C1|F2] Processing fold...
  > [TRAIN] R2: 0.8726 | RMSE: 3478 | MAE: 2126 | Bias: 369.9
  > [VAL]   R2: 0.8769 | RMSE: 3399 | MAE: 2141 | Bias: 293.1

[C1|F3] Processing fold...
  > [TRAIN] R2: 0.8727 | RMSE: 3476 | MAE: 2094 | Bias: 261.5
  > [VAL]   R2: 0.8738 | RMSE: 3442 | MAE: 2070 | Bias: 294.8

[C1|F4] Processing fold...
  > [TRAIN] R2: 0.8742 | RMSE: 3433 | MAE: 2108 | Bias: -88.0
  > [VAL]   R2: 0.8600 | RMSE: 3792 | MAE: 2136 | Bias: -50.5

[C1|F5] Processing fold...
  > [TRAIN] R2: 0.8641 | RMSE: 3586 | MAE: 2139 | Bias: -46.2
  > [VAL]   

,config_id,solver,n_iter_no_change,max_iter,learning_rate_init,learning_rate,hidden_layer_sizes,early_stopping,batch_size,activation,fs_keep_ratio,avg_selected_features,val_rmse,val_mae,val_r2,val_bias,train_mae,train_r2,train_bias,val_med_ae
8,9,adam,25,900,0.005,adaptive,"(256, 128)",True,128,relu,0.8,12.0,3101.456714,1843.090497,0.898393,210.746422,1825.422573,0.901919,203.442629,1177.049962
11,12,adam,25,900,0.005,adaptive,"(128, 64)",True,128,relu,0.8,12.0,3118.955176,1846.342855,0.897351,235.496852,1833.680314,0.899983,232.979915,1174.743361
5,6,adam,25,900,0.005,adaptive,"(128, 64)",True,64,relu,0.8,12.0,3148.648533,1885.337219,0.895311,145.172756,1876.011755,0.897717,144.118332,1216.954420
10,11,adam,25,900,0.010,adaptive,"(128, 64)",True,128,relu,0.8,12.0,3167.808468,1910.170586,0.894099,81.025603,1895.743588,0.896978,81.935894,1239.351469
2,3,adam,25,900,0.005,adaptive,"(256, 128)",True,64,relu,0.8,12.0,3174.677913,1894.666651,0.893633,222.055491,1884.482316,0.896557,216.342468,1215.940946



Best Config found:
{'solver': 'adam', 'n_iter_no_change': 25, 'max_iter': 900, 'learning_rate_init': 0.005, 'learning_rate': 'adaptive', 'hidden_layer_sizes': (256, 128), 'early_stopping': True, 'batch_size': 128, 'activation': 'relu'}


### 3.6. Model with Feature Engineering and Feature Selection - 6
<a id="fe_fs"></a>

In [ ]:


# -------------------------------
# SETTINGS
# -------------------------------
N_ITER = 35
FS_KEEP_RATIO = 0.65
RANDOM_STATE = 42
N_SPLITS = 5
numeric_features = ['year', 'mileage', 'engineSize', 'tax', 'mpg', 'previousOwners']
# Random Forest proxy params para FS
RF_FS_PARAMS = {
    "n_estimators": 500,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "bootstrap": True,
}

# Reprodutibilidade
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# Logging
LOG_FILE = "NN_FS_results.log"
logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format="%(asctime)s - %(message)s",
    filemode="w",
    force=True
)
logging.info(f"- {N_ITER} iterations | {N_SPLITS}-fold | FS_KEEP_RATIO={FS_KEEP_RATIO}")

# Hyperparameter space para MLP
param_distributions = {
    "hidden_layer_sizes": [(256,128), (128,64)],
    "learning_rate_init": [0.05, 0.01, 0.005],
    "activation": ["relu"],
    "solver": ["adam"],
    "batch_size": [64,128],
    "max_iter": [900],
    "learning_rate": ["adaptive"],
    "early_stopping": [True],
    "n_iter_no_change": [25]
}
sampler = ParameterSampler(param_distributions, n_iter=N_ITER, random_state=RANDOM_STATE)

# K-Fold
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# -------------------------------
# RANDOM SEARCH + NN + FS
# -------------------------------
results = []
best_rmse = np.inf
best_config = None

for config_id, params in enumerate(sampler, start=1):
    logging.info(f"######## CONFIG {config_id}/{N_ITER} ########")
    logging.info(f"Params: {params}")

    fold_metrics = []
    fold_selected_feats = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
        X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
        y_train, y_val = y.iloc[train_idx].copy(), y.iloc[val_idx].copy()
        y_train_log = np.log1p(y_train)

        # -------------------------------
        # 1) PREPROCESSING (como o teu pipeline)
        # -------------------------------
        X_train = _normalize_cats(X_train)
        X_val   = _normalize_cats(X_val)

        # Cleaning
        year_state = fit_year_median(X_train, year_col="year", model_col="model")
        X_train = transform_year_with_model_median(X_train, state=year_state)
        X_val   = transform_year_with_model_median(X_val,   state=year_state)

        mileage_state = fit_mileage_imputer(X_train, mileage_col="mileage", do_abs=True)
        X_train = transform_mileage_imputer(X_train, state=mileage_state)
        X_val   = transform_mileage_imputer(X_val,   state=mileage_state)

        engine_state = fit_engine_size_imputer(X_train, engine_col="engineSize")
        X_train = transform_engine_size_imputer(X_train, state=engine_state)
        X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

        X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
        X_val   = transform_tax_custom_rules(X_val,   "tax", "year", "fuelType", "engineSize")

        mpg_state = fit_mpg_imputer(X_train, mpg_col="mpg", do_abs=True)
        X_train = transform_mpg_imputer(X_train, state=mpg_state)
        X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

        owners_state = fit_previous_owners_imputer(X_train, "previousOwners", "year", "mileage")
        X_train = transform_previous_owners_imputer(X_train, state=owners_state)
        X_val   = transform_previous_owners_imputer(X_val,   state=owners_state)

        # Resolver inconsistências
        brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
        X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
        X_val,   _, _ = transform_ambiguous_brands(X_val,   brand_state)

        model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
        X_train, _, _ = transform_invalid_models(X_train, model_state)
        X_val,   _, _ = transform_invalid_models(X_val,   model_state)

        transm_state = fit_transmission_resolver(X_train, valid_transmissions)
        X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
        X_val,   _, _ = transform_transmission_resolver(X_val,   transm_state)

        fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
        X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
        X_val,   _, _ = transform_fueltype_resolver(X_val,   fuel_state)

        # Feature engineering
        X_train = add_owners_flagged(X_train, "previousOwners", "owners_flagged", drop_original=True)
        X_val   = add_owners_flagged(X_val,   "previousOwners", "owners_flagged", drop_original=True)

        X_train = create_age_and_drop_year(X_train, "year", base_year=2020)
        X_val   = create_age_and_drop_year(X_val,   "year", base_year=2020)

        X_train = add_mileage_features(X_train, "mileage", "age", drop_original=True, drop_ratio=True)
        X_val   = add_mileage_features(X_val,   "mileage", "age", drop_original=True, drop_ratio=True)

        X_train = add_engine_bins(X_train, "engineSize", "engine_bin")
        X_val   = add_engine_bins(X_val,   "engineSize", "engine_bin")
        X_train["engine_bin"] = X_train["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")
        X_val["engine_bin"]   = X_val["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")

        low_card_curr = low_card_features + ["engine_bin"]

        # Encoding
        te = MyTargetEncoder(smoothing=5)
        te.fit(X_train[high_card_features], y_train_log)
        X_train_high = te.transform(X_train[high_card_features])
        X_val_high   = te.transform(X_val[high_card_features])

        ohe = MyOneHotEncoder()
        ohe.fit(X_train[low_card_curr])
        X_train_low = ohe.transform(X_train[low_card_curr])
        X_val_low   = ohe.transform(X_val[low_card_curr])

        numeric_base = ["engineSize", "tax", "mpg"]
        engineered_numeric = ["age", "owners_flagged", "log_mileage", "log_miles_per_year"]
        numeric_features_curr = [c for c in (numeric_base + engineered_numeric) if c in X_train.columns]

        X_train_final = pd.concat([X_train[numeric_features_curr], X_train_high, X_train_low], axis=1)
        X_val_final   = pd.concat([X_val[numeric_features_curr],   X_val_high,   X_val_low],   axis=1)

        # Alinhar colunas ANTES do scaling (mais robusto)
        X_val_final = X_val_final.reindex(columns=X_train_final.columns, fill_value=0)

        # Scaling: numéricas + (opcional) target-encoded
        scale_cols = numeric_features_curr + list(X_train_high.columns)  
        scaler = StandardScaler()
        X_train_final[scale_cols] = scaler.fit_transform(X_train_final[scale_cols])
        X_val_final[scale_cols]   = scaler.transform(X_val_final[scale_cols])

        # -------------------------------
        # 2) FEATURE SELECTION (RF proxy)
        # -------------------------------
        n_feats = X_train_final.shape[1]
        k = max(1, min(int(FS_KEEP_RATIO * n_feats), n_feats))

        rf_fs = RandomForestRegressor(**RF_FS_PARAMS)
        rf_fs.fit(X_train_final, y_train_log)
        selector = SelectFromModel(rf_fs, threshold=-np.inf, max_features=k, prefit=True)

        selected_cols = X_train_final.columns[selector.get_support()]
        X_train_sel = X_train_final[selected_cols]
        X_val_sel   = X_val_final[selected_cols]
        fold_selected_feats.append(len(selected_cols))

        # -------------------------------
        # 3) TREINO MLP
        # -------------------------------
        mlp_model = MLPRegressor(
            hidden_layer_sizes=params["hidden_layer_sizes"],
            activation=params.get("activation"),
            solver=params.get("solver"),
            learning_rate_init=params.get("learning_rate_init"),
            learning_rate=params.get("learning_rate"),
            batch_size=params.get("batch_size", "auto"),
            max_iter=params.get("max_iter"),
            n_iter_no_change=params.get("n_iter_no_change", 25),
            early_stopping=params.get("early_stopping"),
            random_state=RANDOM_STATE,
            verbose=False
        )
        mlp_model.fit(X_train_sel, y_train_log)

        y_pred_val = np.expm1(mlp_model.predict(X_val_sel))
        rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))
        fold_metrics.append(rmse_val)

    mean_rmse = np.mean(fold_metrics)
    mean_selected_feats = np.mean(fold_selected_feats)
    logging.info(f"[CONFIG {config_id}] Mean VAL RMSE: {mean_rmse:.3f} | Avg selected feats: {mean_selected_feats:.0f}")

    results.append({"config_id": config_id, **params, "val_rmse": mean_rmse, "avg_selected_feats": mean_selected_feats})

    if mean_rmse < best_rmse:
        best_rmse = mean_rmse
        best_config = {**params}
        logging.info(f"[NEW BEST] CONFIG {config_id} RMSE={best_rmse:.3f}")

# -------------------------------
# 4) RESULTS
# -------------------------------
results_df = pd.DataFrame(results).sort_values("val_rmse")
print(results_df.head(10))
print("\nBest Config:", best_config)

/Users/franciscafernandes/anaconda3/lib/python3.13/site-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 12 is smaller than n_iter=35. Running 12 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/Users/franciscafernandes/anaconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
/Users/franciscafernandes/anaconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


KeyboardInterrupt: 

### 4. Final Configuration
<a id="finalconfig"></a>

In [ ]:

# ==============================================================================
# 0) SETTINGS
# ==============================================================================
RANDOM_STATE = 42

final_params = {
    "min_samples_leaf": 20,
    "max_leaf_nodes": 127,
    "max_iter": 600,
    "max_depth": 25,
    "learning_rate": float(np.float64(0.09105263157894737)),
    "l2_regularization": float(np.float64(1.4473684210526314)),
    "max_bins": 255,
    "random_state": 42,
}

FS_KEEP_RATIO = 0.65
RF_FS_PARAMS = {
    "n_estimators": 500,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "bootstrap": True,
}

print(f"- using params: {final_params}")
print(f"- FS_KEEP_RATIO: {FS_KEEP_RATIO}")

# ==============================================================================
# 1) LOAD TEST + IDS
# ==============================================================================
try:
    test_df_raw = pd.read_csv("test.csv")
except:
    test_df_raw = pd.read_csv("../../project_data/test.csv")

ID_CANDIDATES = ["id", "carID", "carId", "car_id"]
ID_COL = next((c for c in ID_CANDIDATES if c in test_df_raw.columns), None)

if ID_COL is not None:
    test_ids = test_df_raw[ID_COL].copy()
    test_df = test_df_raw.drop(columns=[ID_COL]).copy()
else:
    test_ids = test_df_raw.index.copy()
    test_df = test_df_raw.copy()

# ==============================================================================
# 2) START DATA
# ==============================================================================
X_full = X.copy()
y_full = y.copy()
y_full_log = np.log1p(y_full)

cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]
high_card_features = ["Brand", "model"]
low_card_features = [c for c in categorical_features if c not in high_card_features]

# ==============================================================================
# 4) STRING NORMALIZATION (TRAIN + TEST)
# ==============================================================================
for col in cols_to_normalize:
    if col in X_full.columns:
        X_full[col] = fill_unknown(X_full[col])
        X_full = column_string_transformer(X_full, column=col, remove_middle_spaces=True, allow_extra_chars="")
    if col in test_df.columns:
        test_df[col] = fill_unknown(test_df[col])
        test_df = column_string_transformer(test_df, column=col, remove_middle_spaces=True, allow_extra_chars="")

# Restrict to expected columns (train schema)
expected_cols = [c for c in (numeric_features + categorical_features) if c in X_full.columns]
X_full = X_full[expected_cols].copy()
X_test = test_df.reindex(columns=expected_cols, fill_value=np.nan).copy()

# ==============================================================================
# 5) FIT + TRANSFORM ON FULL TRAIN
# ==============================================================================
print("- fitting transforms on full train")

year_state = fit_year_median(X_full, year_col="year", model_col="model")
X_full = transform_year_with_model_median(X_full, state=year_state)

engine_state = fit_engine_size_imputer(X_full, engine_col="engineSize")
X_full = transform_engine_size_imputer(X_full, state=engine_state)

X_full = transform_tax_custom_rules(X_full, "tax", "year", "fuelType", "engineSize")

mpg_state = fit_mpg_imputer(X_full, mpg_col="mpg", do_abs=True)
X_full = transform_mpg_imputer(X_full, state=mpg_state)

# IMPORTANT: owners imputer must happen BEFORE dropping previousOwners
owners_state = fit_previous_owners_imputer(X_full, "previousOwners", "year", "mileage")
X_full = transform_previous_owners_imputer(X_full, state=owners_state)

brand_state = fit_ambiguous_brand_resolver(X_full, valid_brands)
X_full, _, _ = transform_ambiguous_brands(X_full, brand_state)

model_state = fit_invalid_model_resolver(X_full, valid_models_by_brand)
X_full, _, _ = transform_invalid_models(X_full, model_state)

transm_state = fit_transmission_resolver(X_full, valid_transmissions)
X_full, _, _ = transform_transmission_resolver(X_full, transm_state)

fuel_state = fit_fueltype_resolver(X_full, valid_fueltypes)
X_full, _, _ = transform_fueltype_resolver(X_full, fuel_state)

# ==============================================================================
# 6) FEATURE ENGINEERING (FULL TRAIN)
# ==============================================================================
print("- creating engineered features on full train")

X_full = add_owners_flagged(X_full, owners_col="previousOwners", new_col="owners_flagged", drop_original=True)
X_full = create_age_and_drop_year(X_full, year_col="year", base_year=2020)
X_full = add_mileage_features(X_full, mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True)
X_full = add_engine_bins(X_full, engine_col="engineSize", new_col="engine_bin")

# engine_bin as low-card categorical
X_full["engine_bin"] = X_full["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")
low_card_curr = low_card_features + ["engine_bin"]

# ==============================================================================
# 7) ENCODING (FIT ON FULL TRAIN)
# ==============================================================================
print("- fitting encoders on full train")

te = MyTargetEncoder(smoothing=5)
te.fit(X_full[high_card_features], y_full_log)
X_full_high = te.transform(X_full[high_card_features])

ohe = MyOneHotEncoder()
ohe.fit(X_full[low_card_curr])
X_full_low = ohe.transform(X_full[low_card_curr])

X_full_cat = pd.concat([X_full_high, X_full_low], axis=1)

drop_for_numeric = set(high_card_features + low_card_curr)
numeric_features_curr = [c for c in X_full.columns if c not in drop_for_numeric]

X_full_final = pd.concat([X_full[numeric_features_curr], X_full_cat], axis=1)
print(f"- train matrix shape: {X_full_final.shape}")

# ==============================================================================
# 8) FEATURE SELECTION (FIT ON FULL TRAIN)
# ==============================================================================
print("- fitting feature selector (RF)")

n_feats = X_full_final.shape[1]
k = int(np.ceil(FS_KEEP_RATIO * n_feats))
k = max(1, min(k, n_feats))

rf_fs = RandomForestRegressor(**RF_FS_PARAMS)
rf_fs.fit(X_full_final, y_full_log)

selector = SelectFromModel(
    estimator=rf_fs,
    threshold=-np.inf,
    max_features=k,
    prefit=True
)

selected_cols = X_full_final.columns[selector.get_support()]
X_full_sel = X_full_final[selected_cols]
print(f"- selected features: {len(selected_cols)}/{n_feats}")

# ==============================================================================
# 9) TRAIN FINAL HGB (LOG TARGET)
# ==============================================================================
print("- training final HGB")
hgb_final = HistGradientBoostingRegressor(**final_params)
hgb_final.fit(X_full_sel, y_full_log)
print("- model trained")

# ==============================================================================
# 10) TRANSFORM TEST (TRANSFORM-ONLY) + SAME FE + SAME ENCODING + SAME FS
# ==============================================================================
print("- transforming test set")

X_test = transform_year_with_model_median(X_test, state=year_state)
X_test = transform_engine_size_imputer(X_test, state=engine_state)
X_test = transform_tax_custom_rules(X_test, "tax", "year", "fuelType", "engineSize")
X_test = transform_mpg_imputer(X_test, state=mpg_state)

X_test = transform_previous_owners_imputer(X_test, state=owners_state)

X_test, _, _ = transform_ambiguous_brands(X_test, brand_state)
X_test, _, _ = transform_invalid_models(X_test, model_state)
X_test, _, _ = transform_transmission_resolver(X_test, transm_state)
X_test, _, _ = transform_fueltype_resolver(X_test, fuel_state)

# same FE
X_test = add_owners_flagged(X_test, owners_col="previousOwners", new_col="owners_flagged", drop_original=True)
X_test = create_age_and_drop_year(X_test, year_col="year", base_year=2020)
X_test = add_mileage_features(X_test, mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True)
X_test = add_engine_bins(X_test, engine_col="engineSize", new_col="engine_bin")

X_test["engine_bin"] = X_test["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")

# encoding transform-only
X_test_high = te.transform(X_test[high_card_features])
X_test_low  = ohe.transform(X_test[low_card_curr])
X_test_cat  = pd.concat([X_test_high, X_test_low], axis=1)

drop_for_numeric = set(high_card_features + low_card_curr)
numeric_features_curr_test = [c for c in X_test.columns if c not in drop_for_numeric]

X_test_final = pd.concat([X_test[numeric_features_curr_test], X_test_cat], axis=1)

# align to train columns before selecting
X_test_final = X_test_final.reindex(columns=X_full_final.columns, fill_value=0)

# apply same selected columns
X_test_sel = X_test_final.reindex(columns=selected_cols, fill_value=0)

print(f"- test matrix shape: {X_test_sel.shape}")

# ==============================================================================
# 11) PREDICT + SUBMISSION
# ==============================================================================
pred_log = hgb_final.predict(X_test_sel)
pred_final = np.expm1(pred_log)
pred_final = np.maximum(pred_final, 0)

submission = pd.DataFrame({
    (ID_COL if ID_COL is not None else "id"): test_ids,
    "price": pred_final
})

submission_filename = "submission_hgb_log_target_fs5.csv"
submission.to_csv(submission_filename, index=False)

print(f"- saved: {submission_filename}")
print(submission.head())


### 5. Visualization and Final Regards about the Implementation
<a id="visualization"></a>

In [ ]:
# ---------------------------------------------------------
# 0) CONFIGS GERAIS
# ---------------------------------------------------------
N_SPLITS = 8
N_ITER = 15
RANDOM_STATE = 42

# Percentagem de features a manter no SelectFromModel
FS_KEEP_RATIO = 0.60

# RandomForest para Feature Selection (podes ajustar)
RF_FS_PARAMS = {
    "n_estimators": 500,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "bootstrap": True,
}

# ---------------------------------------------------------
# 0.1) CONFIGURAÇÃO DO LOGGING
# ---------------------------------------------------------

logging.basicConfig(
    filename="random_search_fe_fs_results_60.log",
    level=logging.INFO,
    format="%(asctime)s - %(message)s",
    filemode="w",
    force=True
)
logging.info("- 10 iterations")

# ---------------------------------------------------------
# 0.2) CONFIGURAÇÃO DO RANDOM SEARCH
# ---------------------------------------------------------
param_distributions = {
    "min_samples_leaf": [20, 30, 40],
    "max_leaf_nodes": [31, 63, 127],
    "max_iter": [600, 800, 1000],
    "max_depth": [None, 15, 20, 25],
    "learning_rate": np.linspace(0.08, 0.15, 20),
    "l2_regularization": np.linspace(0.5, 2.5, 20),
    "max_bins": [255],
    "random_state": [RANDOM_STATE],
}

def get_random_configs(n_iter: int):
    configs = []
    keys = list(param_distributions.keys())

    for i in range(n_iter):
        params = {}
        for key in keys:
            params[key] = random.choice(list(param_distributions[key]))

        name = (
            f"R{i}_iter{params['max_iter']}"
            f"_lr{float(params['learning_rate']):.3f}"
            f"_l2{float(params['l2_regularization']):.2f}"
            f"_leaf{params['max_leaf_nodes']}"
        )
        configs.append((name, params))
    return configs

CONFIGS = get_random_configs(N_ITER)

# ---------------------------------------------------------
# 1) K-FOLD E SETUP
# ---------------------------------------------------------
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# Assume que já tens no notebook:
# - X, y
# - numeric_features, categorical_features
# - fill_unknown, column_string_transformer
# - fit_* e transform_* (os teus passos)
# - valid_brands, valid_models_by_brand, valid_transmissions, valid_fueltypes
# - MyTargetEncoder, MyOneHotEncoder

cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]
high_card_features = ["Brand", "model"]
low_card_features  = [c for c in categorical_features if c not in high_card_features]

def _normalize_cats(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in cols_to_normalize:
        if col in df.columns:
            df[col] = fill_unknown(df[col])
            df = column_string_transformer(
                df, column=col, remove_middle_spaces=True, allow_extra_chars=""
            )
    return df

# ---------------------------------------------------------
# 1.1) FEATURE ENGINEERING (FUNÇÕES)
# ---------------------------------------------------------
def create_age_and_drop_year(df, year_col="year", base_year=2020, clip_future=True):
    df = df.copy()
    year = pd.to_numeric(df[year_col], errors="coerce")
    age = base_year - year
    if clip_future:
        age = age.clip(lower=0)
    df["age"] = age
    df = df.drop(columns=[year_col])
    return df

def add_owners_flagged(
    df,
    owners_col="previousOwners",
    new_col="owners_flagged",
    drop_original=True,
    na_as_zero=True
):
    df = df.copy()
    owners = pd.to_numeric(df[owners_col], errors="coerce")

    if na_as_zero:
        owners_filled = owners.fillna(0)
        df[new_col] = (owners_filled > 1).astype("int8")
    else:
        df[new_col] = (owners > 1).fillna(False).astype("int8")

    if drop_original and owners_col in df.columns:
        df = df.drop(columns=[owners_col])

    return df

def add_mileage_features(df, mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True):
    df = df.copy()

    mileage = pd.to_numeric(df[mileage_col], errors="coerce")
    mileage = mileage.where(mileage >= 0, np.nan)

    df["log_mileage"] = np.log1p(mileage)

    age = pd.to_numeric(df[age_col], errors="coerce")
    age_safe = age.clip(lower=1)  # só para divisão
    df["miles_per_year"] = mileage / age_safe
    df["log_miles_per_year"] = np.log1p(df["miles_per_year"])

    if drop_ratio and "miles_per_year" in df.columns:
        df = df.drop(columns=["miles_per_year"])

    if drop_original and mileage_col in df.columns:
        df = df.drop(columns=[mileage_col])

    return df

def add_engine_bins(df, engine_col="engineSize", new_col="engine_bin", bins=None):
    df = df.copy()

    engine = pd.to_numeric(df[engine_col], errors="coerce")
    engine = engine.where(engine >= 0, np.nan)

    if bins is None:
        bins = [0, 1.0, 1.3, 1.6, 2.0, 2.5, 3.0, 4.0, np.inf]

    df[new_col] = pd.cut(engine, bins=bins, include_lowest=True, labels=False).astype("Int64")
    return df

# ---------------------------------------------------------
# 1.2) EVAL DE 1 CONFIG
# ---------------------------------------------------------
def eval_one_config(name: str, params: dict) -> dict:
    fold_rows = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
        X_train = X.iloc[train_idx].copy()
        X_val   = X.iloc[val_idx].copy()
        y_train = y.iloc[train_idx].copy()
        y_val   = y.iloc[val_idx].copy()

        # TARGET (LOG)
        y_train_log = np.log1p(y_train)

        # Normalizar categóricas base
        X_train = _normalize_cats(X_train)
        X_val   = _normalize_cats(X_val)

        # Restringir às colunas base (mantém previousOwners e mileage para cleaning)
        base_cols = [c for c in (numeric_features + categorical_features) if c in X_train.columns]
        X_train = X_train[base_cols].copy()
        X_val   = X_val[base_cols].copy()

        # -----------------------------------------------------
        # A) CLEANING
        # -----------------------------------------------------
        year_state = fit_year_median(X_train, year_col="year", model_col="model")
        X_train = transform_year_with_model_median(X_train, state=year_state)
        X_val   = transform_year_with_model_median(X_val,   state=year_state)

        engine_state = fit_engine_size_imputer(X_train, engine_col="engineSize")
        X_train = transform_engine_size_imputer(X_train, state=engine_state)
        X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

        X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
        X_val   = transform_tax_custom_rules(X_val,   "tax", "year", "fuelType", "engineSize")

        mpg_state = fit_mpg_imputer(X_train, mpg_col="mpg", do_abs=True)
        X_train = transform_mpg_imputer(X_train, state=mpg_state)
        X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

        owners_state = fit_previous_owners_imputer(X_train, "previousOwners", "year", "mileage")
        X_train = transform_previous_owners_imputer(X_train, state=owners_state)
        X_val   = transform_previous_owners_imputer(X_val,   state=owners_state)

        # -----------------------------------------------------
        # B) RESOLVERS
        # -----------------------------------------------------
        brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
        X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
        X_val,   _, _ = transform_ambiguous_brands(X_val,   brand_state)

        model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
        X_train, _, _ = transform_invalid_models(X_train, model_state)
        X_val,   _, _ = transform_invalid_models(X_val,   model_state)

        transm_state = fit_transmission_resolver(X_train, valid_transmissions)
        X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
        X_val,   _, _ = transform_transmission_resolver(X_val,   transm_state)

        fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
        X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
        X_val,   _, _ = transform_fueltype_resolver(X_val,   fuel_state)

        # -----------------------------------------------------
        # C) FEATURE ENGINEERING
        # -----------------------------------------------------
        X_train = add_owners_flagged(X_train, owners_col="previousOwners", new_col="owners_flagged", drop_original=True)
        X_val   = add_owners_flagged(X_val,   owners_col="previousOwners", new_col="owners_flagged", drop_original=True)

        X_train = create_age_and_drop_year(X_train, year_col="year", base_year=2020)
        X_val   = create_age_and_drop_year(X_val,   year_col="year", base_year=2020)

        X_train = add_mileage_features(X_train, mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True)
        X_val   = add_mileage_features(X_val,   mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True)

        X_train = add_engine_bins(X_train, engine_col="engineSize", new_col="engine_bin")
        X_val   = add_engine_bins(X_val,   engine_col="engineSize", new_col="engine_bin")

        # engine_bin como low-card para OHE (robusto a NaNs)
        X_train["engine_bin"] = X_train["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")
        X_val["engine_bin"]   = X_val["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")

        low_card_curr = low_card_features + ["engine_bin"]

        # -----------------------------------------------------
        # D) ENCODING
        # -----------------------------------------------------
        te = MyTargetEncoder(smoothing=5)
        te.fit(X_train[high_card_features], y_train_log)
        X_train_high = te.transform(X_train[high_card_features])
        X_val_high   = te.transform(X_val[high_card_features])

        ohe = MyOneHotEncoder()
        ohe.fit(X_train[low_card_curr])
        X_train_low = ohe.transform(X_train[low_card_curr])
        X_val_low   = ohe.transform(X_val[low_card_curr])

        X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
        X_val_cat   = pd.concat([X_val_high,   X_val_low],   axis=1)

        drop_for_numeric = set(high_card_features + low_card_curr)
        numeric_features_curr = [c for c in X_train.columns if c not in drop_for_numeric]

        X_train_final = pd.concat([X_train[numeric_features_curr], X_train_cat], axis=1)
        X_val_final   = pd.concat([X_val[numeric_features_curr],   X_val_cat],   axis=1)

        # alinhar colunas
        X_val_final = X_val_final.reindex(columns=X_train_final.columns, fill_value=0)

        # -----------------------------------------------------
        # D.1) FEATURE SELECTION (SelectFromModel + RandomForest)
        #     - threshold=-np.inf
        #     - max_features = 80% das features
        # -----------------------------------------------------
        n_feats = X_train_final.shape[1]
        k = int(np.ceil(FS_KEEP_RATIO * n_feats))
        k = max(1, min(k, n_feats))

        rf_fs = RandomForestRegressor(**RF_FS_PARAMS)
        rf_fs.fit(X_train_final, y_train_log)

        selector = SelectFromModel(
            estimator=rf_fs,
            threshold=-np.inf,
            max_features=k,
            prefit=True
        )

        support = selector.get_support()
        selected_cols = X_train_final.columns[support]

        X_train_sel = X_train_final[selected_cols]
        X_val_sel   = X_val_final[selected_cols]

        # -----------------------------------------------------
        # E) TREINO (HGB)
        # -----------------------------------------------------
        model = HistGradientBoostingRegressor(**params)
        model.fit(X_train_sel, y_train_log)

        pred_tr_log  = model.predict(X_train_sel)
        pred_val_log = model.predict(X_val_sel)

        pred_tr  = np.expm1(pred_tr_log)
        pred_val = np.expm1(pred_val_log)

        mae_tr  = mean_absolute_error(y_train, pred_tr)
        mae_val = mean_absolute_error(y_val,   pred_val)

        fold_rows.append({
            "config": name,
            "fold": fold,
            "n_features_total": n_feats,
            "n_features_selected": len(selected_cols),
            "train_mae": mae_tr,
            "val_mae": mae_val
        })

    df_folds = pd.DataFrame(fold_rows)
    mean_mae_tr  = df_folds["train_mae"].mean()
    mean_mae_val = df_folds["val_mae"].mean()
    mean_sel     = df_folds["n_features_selected"].mean()

    print(f"   >> [{name}] Train MAE: {mean_mae_tr:.2f} | Val MAE: {mean_mae_val:.2f} | Sel: {mean_sel:.0f} feats")
    logging.info(
        f"{name} | Train MAE: {mean_mae_tr:.4f} | Val MAE: {mean_mae_val:.4f} | "
        f"AvgSelected: {mean_sel:.1f} | Params: {params}"
    )

    out = {
        "config": name,
        "val_mae_mean": mean_mae_val,
        "train_mae_mean": mean_mae_tr,
        "avg_selected_features": float(mean_sel),
        **params
    }
    return out

# ---------------------------------------------------------
# 2) CORRER TODAS AS CONFIGS
# ---------------------------------------------------------
results = []
for i, (name, params) in enumerate(CONFIGS):
    if i % 5 == 0:
        print(f"\n-- doing {i+1}/{len(CONFIGS)}")

    res = eval_one_config(name, params)
    results.append(res)

results_df = pd.DataFrame(results)

# ---------------------------------------------------------
# 3) RESULTADOS FINAIS
# ---------------------------------------------------------
print("\n- 10 BEST Configurations")
results_df_sorted = results_df.sort_values("val_mae_mean", ascending=True)

cols_show = [
    "config",
    "val_mae_mean", "train_mae_mean",
    "avg_selected_features",
    "learning_rate", "max_iter", "l2_regularization", "max_leaf_nodes", "max_depth", "min_samples_leaf"
]

display(results_df_sorted[cols_show].head(10))

results_df_sorted.to_csv("random_search_fe_fs_results_60.csv", index=False)
print("stored at: random_search_fe_fs_results_60.csv")
print(" log in: random_search_fe_fs_results_60.log")
